# Green Space, Urban Heat, and Residential Equity in Metropolitan Melbourne

## 1. Introduction

This project investigates the spatial relationship between green space, urban heat, and residential equity in Melbourne. The main purpose is to examine whether open space and vegetation-related environmental benefits are evenly distributed, whether areas with more green space tend to experience lower heat exposure, and whether these environmental benefits are shared fairly across different social groups.

The analysis uses SA2 as the main spatial unit. Environmental indicators, including open space and urban heat, are combined with ABS Census variables to support comparison across different residential areas.

## 2. Research Question

How are green space, urban heat exposure, and residential equity spatially related across Melbourne’s metropol

## 3. study area

The study area is Metropolitan Melbourne. The main spatial analysis unit is SA2. All environmental and social indicators are aggregated to the SA2 level for comparison, mapping, and priority analysis.iority analysis.and mapping.and mapping.

In [2]:
# Import libraries and set project paths

from pathlib import Path
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

# Find the project root folder automatically
def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "data" / "raw").exists():
            return path
    raise FileNotFoundError("Project root folder was not found. Please check the folder structure.")

project_dir = find_project_root()

# Main folders
raw_dir = project_dir / "data" / "raw"
processed_dir = project_dir / "data" / "processed"
output_dir = project_dir / "outputs"

# Raw data folders
boundary_dir = raw_dir / "boundary"
census_dir = raw_dir / "census"
open_space_dir = raw_dir / "open_space"
urban_heat_dir = raw_dir / "urban_heat"

# Output folders
map_dir = output_dir / "maps"
chart_dir = output_dir / "charts"
table_dir = output_dir / "tables"

# Create output folders if they do not exist
for folder in [processed_dir, map_dir, chart_dir, table_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directory:", project_dir)
print("Raw data folder:", raw_dir)
print("Processed data folder:", processed_dir)
print("Output folder:", output_dir)

Project directory: C:\Users\15483\Documents\GitHub\geom90006_a4
Raw data folder: C:\Users\15483\Documents\GitHub\geom90006_a4\data\raw
Processed data folder: C:\Users\15483\Documents\GitHub\geom90006_a4\data\processed
Output folder: C:\Users\15483\Documents\GitHub\geom90006_a4\outputs


In [12]:
# Load main datasets

greater_melb = gpd.read_file(boundary_dir / "greater_melbourne_boundary.gpkg")
sa2_full = gpd.read_file(boundary_dir / "sa2_greater_melbourne.gpkg")

# Final SA2 analysis area
sa2_analysis = gpd.read_file(boundary_dir / "sa2_analysis_area.gpkg")

open_space = gpd.read_file(open_space_dir / "open_space_melbourne.gpkg")
urban_heat = gpd.read_file(urban_heat_dir / "urban_heat.gpkg")

print("Greater Melbourne boundary:", greater_melb.shape)
print("Full SA2 boundary:", sa2_full.shape)
print("Final SA2 analysis area:", sa2_analysis.shape)
print("Open space:", open_space.shape)
print("Urban heat:", urban_heat.shape)

Greater Melbourne boundary: (1, 12)
Full SA2 boundary: (361, 18)
Final SA2 analysis area: (362, 18)
Open space: (38639, 22)
Urban heat: (55603, 20)


In [13]:
# Correct the final SA2 analysis area
# Keep original SA2 polygons that intersect with the urban heat data

TARGET_CRS = "EPSG:7855"

sa2_full_7855 = sa2_full.to_crs(TARGET_CRS)
urban_heat_7855 = urban_heat.to_crs(TARGET_CRS)

# Spatial join: select SA2s that intersect with urban heat data
sa2_joined = gpd.sjoin(
    sa2_full_7855,
    urban_heat_7855[["geometry"]],
    how="inner",
    predicate="intersects"
)

# Keep unique SA2 polygons
sa2_analysis = sa2_full_7855.loc[sa2_joined.index.unique()].copy()

print("Corrected final SA2 analysis area:", sa2_analysis.shape)

Corrected final SA2 analysis area: (360, 18)


In [14]:
# Save corrected SA2 analysis area

sa2_analysis.to_file(D:\BaiduNetdiskDownload / "sa2_analysis_area1.gpkg", driver="GPKG")

print("Corrected sa2_analysis_area.gpkg has been saved.")

SyntaxError: invalid syntax (3535913496.py, line 3)

## 4. Data Sources

The project will use several spatial and non-spatial datasets, including:

- ABS SA2 boundary data
- Greater Melbourne boundary data
- ABS Census data
- Open space polygon data
- Urban tree point data
- Landsat LST / NDVI or UHI data

These datasets support the analysis of green space distribution, urban heat exposure, and social equity.

## 5. Data Preprocessing

The preprocessing steps include:

1. Loading spatial datasets
2. Checking coordinate reference systems
3. Reprojecting all spatial data to a common projected CRS
4. Clipping datasets to Greater Melbourne
5. Aggregating indicators to the SA2 level
6. Creating a master SA2 dataset for later analysis